In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import itertools
import tensorflow as tf
import os

# Fix random seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load and prepare data
csv_path = 'all_temperature_cleaned.csv'
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()
df['datetime'] = pd.to_datetime(df['timestamp'] + ' ' + df['time'], format='%Y-%m-%d %H:%M')

region = 'Rakhiyal'  # Change or loop over regions if needed

# Aggregate to daily average temperature
df.set_index('datetime', inplace=True)
df_daily = df[[region]].resample('D').mean()
df_daily[region] = df_daily[region].interpolate(method='time')

# Normalize data
scaler = MinMaxScaler(feature_range=(0, 1))
scaled_data = scaler.fit_transform(df_daily)

# Function to create sequences
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

# Split train/test based on date
dates = df_daily.index
train_mask = (dates.year >= 2019) & (dates.year <= 2023)
test_mask = (dates.year == 2024)

# Hyperparameter grid
seq_lengths = [15, 30]
lstm_units_list = [32, 50]
batch_sizes = [16, 32]
epochs_list = [20, 50]

param_grid = list(itertools.product(seq_lengths, lstm_units_list, batch_sizes, epochs_list))

results = []

# Create output folder if needed
output_folder = 'trial'
os.makedirs(output_folder, exist_ok=True)

for seq_length, lstm_units, batch_size, epochs in param_grid:
    # Prepare sequences
    X, y = create_sequences(scaled_data, seq_length)
    dates_seq = dates[seq_length:]

    # Apply masks to sequences (adjusted for sequence length)
    train_indices = train_mask[seq_length:]
    test_indices = test_mask[seq_length:]

    X_train = X[train_indices]
    y_train = y[train_indices]
    X_test = X[test_indices]
    y_test = y[test_indices]

    # Reshape for LSTM: (samples, timesteps, features)
    X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
    X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

    # Build model
    model = Sequential([
        LSTM(lstm_units, activation='relu', input_shape=(seq_length, 1)),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')

    # Train model
    model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, verbose=0)

    # Predict and inverse scale
    y_pred = model.predict(X_test)
    y_test_inv = scaler.inverse_transform(y_test)
    y_pred_inv = scaler.inverse_transform(y_pred)

    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))

    print(f"seq_len={seq_length}, units={lstm_units}, batch={batch_size}, epochs={epochs} => RMSE: {rmse:.4f}")

    results.append({
        'seq_length': seq_length,
        'lstm_units': lstm_units,
        'batch_size': batch_size,
        'epochs': epochs,
        'rmse': rmse
    })

# Save results
results_df = pd.DataFrame(results)
results_csv_path = os.path.join(output_folder, 'lstm_hyperparam_tuning_results.csv')
results_df.to_csv(results_csv_path, index=False)

# Report best parameters
best_row = results_df.loc[results_df['rmse'].idxmin()]
print("\nBest LSTM parameters:")
print(best_row)


C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
seq_len=15, units=32, batch=16, epochs=20 => RMSE: 0.8918


C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
seq_len=15, units=32, batch=16, epochs=50 => RMSE: 0.8184


C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
seq_len=30, units=50, batch=16, epochs=50 => RMSE: 0.8523


C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
seq_len=30, units=50, batch=32, epochs=20 => RMSE: 1.1433


C:\Users\PARIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
seq_len=30, units=50, batch=32, epochs=50 => RMSE: 0.8629

Best LSTM parameters:
seq_length    15.000000
lstm_units    50.000000
batch_size    32.000000
epochs        50.000000
rmse           0.809273
Name: 7, dtype: float64
